# Davinci Notebook 12 — SFT Training, Chat Formatting & Tool Traces

Canonical chat serialization, assistant-only target masking, host-compatible tool-call message structure, training/validation splitting, and a fine-tuning loop. Special tokens must be added to the frozen tokenizer before training; this notebook's placeholder tokens are not automatically registered.


In [ ]:
import json, random, torch, torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
random.seed(42)


## 1. Canonical message structure and serialization
Use a consistent serialization strategy for training and inference. The assistant tool-call payload follows the strict JSON protocol in `src/davinci/tool_call.py`: exactly `name` and `arguments`. The host executes only registered tools; tool observations use the `tool` role and are not assistant-authored targets.


In [ ]:
SPECIAL={'user':'<|user|>','assistant':'<|assistant|>','system':'<|system|>','tool':'<|tool|>','end':'<|end|>','turn_end':'<|turn_end|>','call':'<|tool_call|>','result':'<|tool_result|>'}
def serialize(messages):
 out=[]
 for m in messages:
  role=m.get('role'); content=m.get('content','')
  if role not in SPECIAL: raise ValueError(f'unsupported role: {role}')
  out.append(SPECIAL[role]+str(content)+SPECIAL['end'])
 return ''.join(out)+SPECIAL['turn_end']

def tool_call_content(name,arguments):
 return SPECIAL['call']+json.dumps({'name':name,'arguments':arguments},ensure_ascii=False,separators=(',',':'))

def tool_result_content(name,result):
 return SPECIAL['result']+json.dumps({'name':name,'result':result},ensure_ascii=False,separators=(',',':'))

example_trace=[
 {'role':'user','content':'What is 19 + 23?'},
 {'role':'assistant','content':tool_call_content('add',{'left':19,'right':23})},
 {'role':'tool','content':tool_result_content('add',{'sum':42})},
 {'role':'assistant','content':'42'},
]
serialized_example=serialize(example_trace)
assert SPECIAL['call'] in serialized_example and SPECIAL['result'] in serialized_example
print(serialized_example)


## 2. Assistant-only masking dataset
The caller must supply a tokenizer that can tokenize the serialized messages and identify the token boundaries. This template masks non-assistant tokens with `-100`; adapt it to the actual tokenizer's special token IDs and turn boundaries.

In [ ]:
def build_masked_example(messages,tokenizer):
 ids=[]; labels=[]
 for m in messages:
  chunk=tokenizer(SPECIAL[m['role']]+m.get('content','')+SPECIAL['end'],add_special_tokens=False)['input_ids']
  ids.extend(chunk); labels.extend(chunk if m['role']=='assistant' else [-100]*len(chunk))
 tail=tokenizer(SPECIAL['turn_end'],add_special_tokens=False)['input_ids']; ids.extend(tail); labels.extend(tail)
 return {'input_ids':ids,'labels':labels}
print('Masking template defined; tokenizer-specific behavior still requires testing.')


## 3. Dataset split and collator
Split by example ID or source document before training; avoid overlapping related examples across splits. This basic collator pads and uses labels=-100 for padding.

In [ ]:
def split_examples(examples,val_fraction=.05,seed=42):
 ids=list(range(len(examples))); random.Random(seed).shuffle(ids); n=max(1,int(len(ids)*val_fraction)); val=set(ids[:n]); return [x for i,x in enumerate(examples) if i not in val],[x for i,x in enumerate(examples) if i in val]
def collate_masked(batch,pad_id):
 maxlen=max(len(x['input_ids']) for x in batch); inp=[]; lab=[]
 for x in batch:
  pad=maxlen-len(x['input_ids']); inp.append(x['input_ids']+[pad_id]*pad); lab.append(x['labels']+[-100]*pad)
 return torch.tensor(inp),torch.tensor(lab)


## 4. SFT loop (generic model interface)
Assumes model(input_ids) returns logits `[B,T,V]`. This is a baseline loop; it omits packed-example masking, multi-GPU, and production scheduler/checkpoint policies.

In [ ]:
def sft_train_step(model,batch,optimizer,device,clip=1.0):
 ids,labels=[x.to(device) for x in batch]; model.train(); optimizer.zero_grad(set_to_none=True); logits=model(ids)
 # For causal models, shift logits and labels. Input labels are aligned to token positions.
 loss=F.cross_entropy(logits[:,:-1,:].float().reshape(-1,logits.size(-1)),labels[:,1:].reshape(-1),ignore_index=-100)
 loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),clip); optimizer.step(); return float(loss.detach())


## 5. Dataset audit before fine-tuning
- Check source licenses and redistribution conditions.
- Deduplicate against pretraining and held-out data.
- Review generated reasoning for correctness and leakage.
- Remove personally identifying or confidential content.
- Keep tool traces structured and separate observations from model-authored messages.
- Evaluate separately on Persian, English, bilingual and tool-use tasks.


## 6. Tool-trace protocol alignment
Before training, parse every assistant tool-call payload with the same strict protocol as the inference runtime, validate tool names/arguments against the tool manifest, and check that each tool result corresponds to a preceding call. Do not train the model to emit host-only secrets, stack traces, or unredacted tool output. Tool-call assistant messages are supervised targets; tool observations remain context only in the assistant-only masking template above.
